# 03 — Evaluation

**But** : charger le meilleur checkpoint depuis le disque et évaluer le modèle sur le test set.

> Ce notebook est **totalement indépendant** du notebook d'entraînement.
> Il ne dépend d'aucune variable en mémoire — il recharge tout depuis les fichiers sauvegardés.

## 1. Setup

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    %cd /content/drive/MyDrive/LAAFI_AI
except ImportError:
    pass

import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

In [ ]:
import datasets
datasets.logging.set_verbosity_error()

import torch
from laafi_ai.cli_train import set_seed
from laafi_ai.config import ExperimentConfig
from laafi_ai.data import PCamDataModule
from laafi_ai.evaluate import collect_predictions
from laafi_ai.evaluation_report import generate_full_report
from laafi_ai.inference import load_model_from_checkpoint
from laafi_ai.logging_utils import setup_logging
from laafi_ai.metrics import compute_binary_metrics
from laafi_ai.model import get_device
from laafi_ai.paths import resolve_project_paths

setup_logging()

## 2. Configuration et chemins

In [ ]:
config = ExperimentConfig.from_yaml('configs/default.yaml')
config.output_dir = 'outputs_finetune_layer4'

set_seed(config.seed)
device = get_device(config.device)
paths = resolve_project_paths(config, project_root=PROJECT_ROOT)

BEST_CHECKPOINT = paths.checkpoint_dir / 'best_resnet50_pcam.pt'
print('Checkpoint:', BEST_CHECKPOINT)
print('Exists:', BEST_CHECKPOINT.exists())

## 3. Charger le modèle depuis le checkpoint

In [ ]:
model, loaded_config = load_model_from_checkpoint(BEST_CHECKPOINT, device)
print('Modèle chargé ✓')
print('Architecture:', loaded_config.model.architecture)

## 4. Charger le test set

In [ ]:
data_module = PCamDataModule(config.data)
_, _, test_loader = data_module.dataloaders()
print(f'Test set: {len(test_loader.dataset):,} images')

## 5. Collecter les prédictions

In [ ]:
import numpy as np

labels, probabilities = collect_predictions(model, test_loader, device)

# Sauvegarder pour réutilisation future
np.save(paths.predictions_dir / 'test_labels.npy', labels)
np.save(paths.predictions_dir / 'test_probs.npy', probabilities)
print(f'Prédictions collectées et sauvegardées ({len(labels):,} exemples)')

## 6. Calculer les métriques

In [ ]:
threshold = config.training.decision_threshold
metrics = compute_binary_metrics(labels, probabilities, threshold=threshold)

print(f'\n--- Métriques Test (seuil = {threshold}) ---')
print(f'  AUC:               {metrics.auc:.4f}')
print(f'  Accuracy:          {metrics.accuracy:.4f}')
print(f'  Sensibilité:       {metrics.sensitivity:.4f}')
print(f'  Spécificité:       {metrics.specificity:.4f}')
print(f'  Precision:         {metrics.precision:.4f}')
print(f'  Average Precision: {metrics.average_precision:.4f}')

## 7. Générer le rapport complet (figures + CSV)

In [ ]:
generate_full_report(
    labels=labels,
    probabilities=probabilities,
    metrics=metrics,
    figures_dir=paths.figures_dir,
    metrics_dir=paths.metrics_dir,
    threshold=threshold,
)
print('\nRapport complet généré :')
print(f'  {paths.figures_dir / "roc_curve.png"}')
print(f'  {paths.figures_dir / "pr_curve.png"}')
print(f'  {paths.figures_dir / "confusion_matrix.png"}')
print(f'  {paths.metrics_dir / "metrics_finales.csv"}')

## 8. Afficher les figures

In [ ]:
from IPython.display import display, Image as IPImage

for fig_name in ['roc_curve.png', 'pr_curve.png', 'confusion_matrix.png']:
    fig_path = paths.figures_dir / fig_name
    if fig_path.exists():
        print(f'\n{fig_name}')
        display(IPImage(filename=str(fig_path)))

---
**Prochain notebook** : `04_inference.ipynb` pour des prédictions individuelles et Grad-CAM.